# OLMo 7B in DeepChem: Generation, Fine-Tuning & Pre-Training

**Author:** GSoC Contributor  
**Mentors:** Riya, Harindhar  
**Project:** LLM support for 7B models in DeepChem  
**Model:** [allenai/OLMo-7B](https://huggingface.co/allenai/OLMo-7B)

---

This tutorial demonstrates the complete workflow for using **OLMo** (Open Language Model by Allen AI) inside DeepChem for:

1. **Molecular generation** – unconditional and prompt-conditioned SMILES generation  
2. **Classification** – binary property prediction (e.g. BBBP blood-brain barrier)  
3. **Regression** – continuous property prediction (e.g. lipophilicity / logP)  
4. **Continued pre-training** – adapting OLMo to a molecular corpus  

### Prerequisites

```bash
pip install deepchem transformers>=4.40.0 torch accelerate peft rdkit-pypi
```

For 7B inference on a single GPU you will need ≥ 16 GB VRAM (fp16) or a machine with sufficient CPU RAM for offloading.
For development and this tutorial we use `allenai/OLMo-1B` which fits on most modern GPUs.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

import deepchem as dc
print(f'DeepChem version: {dc.__version__}')

## 1. Imports

All OLMo-related classes live in `deepchem.models.torch_models`.

In [ ]:
from deepchem.models.torch_models.olmo import OLMoModel
from deepchem.models.torch_models.olmo_generation import (
    OLMoGenerationFeaturizer,
    OLMoPretrainer,
    MolecularTextDataset,
    evaluate_generation,
)
from deepchem.models.torch_models.olmo_finetune import (
    OLMoFineTuner,
    OLMoSupervisedDataset,
    evaluate_classification,
    evaluate_regression,
)

## 2. Loading the Model

We use `allenai/OLMo-1B` for this notebook. Switch to `allenai/OLMo-7B` for full-scale experiments.

```python
MODEL_NAME = "allenai/OLMo-7B"  # production
```

In [ ]:
MODEL_NAME = "allenai/OLMo-1B"   # change to OLMo-7B for full experiment

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = "bfloat16" if (torch.cuda.is_available() and
                        torch.cuda.get_device_capability()[0] >= 8) else "float32"

print(f'Using device : {DEVICE}')
print(f'Using dtype  : {DTYPE}')

## 3. Part A – Molecular Generation

### 3.1 Initialise a generation model

In [ ]:
gen_model = OLMoModel(
    task="generation",
    model_name=MODEL_NAME,
    max_seq_length=256,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
print(gen_model)

### 3.2 Unconditional generation

We generate molecules from an empty prompt – the model samples from its prior over SMILES.

In [ ]:
unconditional = gen_model.generate_molecules(
    prompt="",
    n_molecules=10,
    max_new_tokens=100,
    temperature=1.0,
    top_p=0.95,
)

print("Unconditionally generated molecules:")
for i, mol in enumerate(unconditional, 1):
    print(f"  {i:2d}. {mol}")

### 3.3 Prompt-conditioned generation

We condition on the beginning of an aspirin SMILES.

In [ ]:
conditional = gen_model.generate_molecules(
    prompt="CC(=O)Oc1ccc",   # partial aspirin scaffold
    n_molecules=5,
    max_new_tokens=80,
    temperature=0.8,
)

print("Prompt-conditioned molecules (aspirin scaffold):")
for m in conditional:
    print(f"  CC(=O)Oc1ccc{m}")

### 3.4 Validity check with RDKit

In [ ]:
try:
    from rdkit import Chem

    def check_validity(smiles_list):
        valid = [s for s in smiles_list if Chem.MolFromSmiles(s) is not None]
        print(f"  Valid: {len(valid)}/{len(smiles_list)} "
              f"({100*len(valid)/max(len(smiles_list),1):.1f}%)")
        return valid

    print("Unconditional validity:")
    valid_unc = check_validity(unconditional)
    print("Conditional validity:")
    valid_cond = check_validity(conditional)

except ImportError:
    print("rdkit not installed – skipping validity check.")

---
## 4. Part B – Classification Fine-Tuning (BBBP)

We fine-tune OLMo on the **Blood-Brain Barrier Penetration** (BBBP) dataset, a standard benchmark in molecular property prediction.

### 4.1 Load & inspect the dataset

In [ ]:
# Load BBBP via MoleculeNet
tasks_bbbp, datasets_bbbp, _ = dc.molnet.load_bbbp(
    featurizer='Raw',
    splitter='scaffold'
)
train_bbbp, val_bbbp, test_bbbp = datasets_bbbp

print(f"Train size : {len(train_bbbp)}")
print(f"Val   size : {len(val_bbbp)}")
print(f"Test  size : {len(test_bbbp)}")
print(f"Tasks      : {tasks_bbbp}")
print(f"Class balance (train): {train_bbbp.y.mean():.2%} positive")

### 4.2 Initialise a classification model with LoRA

In [ ]:
cls_model = OLMoModel(
    task="classification",
    model_name=MODEL_NAME,
    n_tasks=1,
    n_classes=2,
    use_lora=True,
    lora_r=8,
    lora_alpha=16,
    max_seq_length=256,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)

n_params = cls_model.get_num_trainable_params()
print(f"Trainable parameters: {n_params:,}")

### 4.3 Fine-tune

In [ ]:
tuner_cls = OLMoFineTuner(
    cls_model,
    output_dir="./olmo_bbbp_ckpt",
    learning_rate=2e-4,
    batch_size=8,
    max_epochs=5,
    grad_accum_steps=4,
    loss_fn="bce",
    save_best=True,
    early_stopping_patience=3,
    fp16=(DTYPE == "float16"),
    bf16=(DTYPE == "bfloat16"),
)

history_cls = tuner_cls.fit_deepchem(
    train_dc=train_bbbp,
    val_dc=val_bbbp,
    max_length=256,
    smiles_prefix="Predict BBBP: ",
)

print("Training complete!")
print(f"Final train loss : {history_cls['train_loss'][-1]:.4f}")
if history_cls['val_metric']:
    print(f"Best val AUROC   : {max(history_cls['val_metric']):.4f}")

### 4.4 Loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_cls['train_loss'], label='Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('BBBP Classification – Training Loss')
axes[0].legend()

if history_cls['val_metric']:
    axes[1].plot(history_cls['val_metric'], color='orange', label='Val AUROC')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUROC')
    axes[1].set_title('BBBP Classification – Validation AUROC')
    axes[1].legend()

plt.tight_layout()
plt.savefig('bbbp_training_curves.png', dpi=150)
plt.show()

### 4.5 Evaluate on held-out test set

In [ ]:
test_smiles = [str(s) for s in test_bbbp.X.flatten()]
test_labels = test_bbbp.y

metrics_cls = evaluate_classification(
    cls_model,
    test_smiles,
    test_labels,
    max_length=256,
    batch_size=16,
)

print("\n=== BBBP Test Set Results ===")
for k, v in metrics_cls.items():
    print(f"  {k:12s}: {v:.4f}")

---
## 5. Part C – Regression Fine-Tuning (Lipophilicity)

We predict **experimental lipophilicity (logD7.4)** using the Lipophilicity dataset.

### 5.1 Load dataset

In [ ]:
tasks_lipo, datasets_lipo, _ = dc.molnet.load_lipo(
    featurizer='Raw', splitter='scaffold'
)
train_lipo, val_lipo, test_lipo = datasets_lipo

print(f"Train size : {len(train_lipo)}")
print(f"Tasks      : {tasks_lipo}")
print(f"Label range: [{train_lipo.y.min():.2f}, {train_lipo.y.max():.2f}]")

### 5.2 Initialise regression model

In [ ]:
reg_model = OLMoModel(
    task="regression",
    model_name=MODEL_NAME,
    n_tasks=1,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    max_seq_length=256,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
print(f"Trainable params: {reg_model.get_num_trainable_params():,}")

### 5.3 Fine-tune with Huber loss

In [ ]:
tuner_reg = OLMoFineTuner(
    reg_model,
    output_dir="./olmo_lipo_ckpt",
    learning_rate=1e-4,
    batch_size=8,
    max_epochs=5,
    loss_fn="huber",
    save_best=True,
    early_stopping_patience=3,
    bf16=(DTYPE == "bfloat16"),
)

history_reg = tuner_reg.fit_deepchem(
    train_dc=train_lipo,
    val_dc=val_lipo,
    max_length=256,
    smiles_prefix="Predict logD: ",
)

print(f"Final train loss : {history_reg['train_loss'][-1]:.4f}")
if history_reg['val_metric']:
    print(f"Best val RMSE    : {min(history_reg['val_metric']):.4f}")

### 5.4 Evaluate on test set

In [ ]:
test_smiles_lipo = [str(s) for s in test_lipo.X.flatten()]

metrics_reg = evaluate_regression(
    reg_model,
    test_smiles_lipo,
    test_lipo.y,
    max_length=256,
)

print("\n=== Lipophilicity Test Set Results ===")
for k, v in metrics_reg.items():
    print(f"  {k:6s}: {v:.4f}")

---
## 6. Part D – Continued Pre-Training on Molecular Data

Here we show how to adapt a pre-trained OLMo checkpoint to a molecular corpus using the causal language modelling objective.  
In practice you would use a large SMILES corpus (e.g. ZINC-20, PubChem, ChEMBL).

### 6.1 Prepare a toy molecular corpus

In [ ]:
# In production: load millions of SMILES from ZINC / PubChem
sample_corpus = [
    "CC(=O)Oc1ccccc1C(=O)O",          # Aspirin
    "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",  # Testosterone
    "c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34",  # Pyrene
    "CC(C)NCC(O)c1ccc(O)c(O)c1",      # Isoprenaline
    "OC(=O)c1ccc(O)cc1",              # 4-Hydroxybenzoic acid
    "C1CC2=CC=CC=C2N1",               # 1,2,3,4-Tetrahydroisoquinoline
    "CC(=O)Nc1ccc(O)cc1",             # Paracetamol
    "C[N+]1=CC=CC=C1.F[B-](F)(F)F",  # 1-Methylpyridinium tetrafluoroborate
    "O=C1CCCN1",                       # 2-Pyrrolidinone
    "c1ccc(CCN)cc1",                  # 2-Phenylethylamine
]

print(f"Corpus size: {len(sample_corpus)} molecules")

### 6.2 Instantiate pre-trainer

In [ ]:
pretrain_model = OLMoModel(
    task="generation",
    model_name=MODEL_NAME,
    max_seq_length=128,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)

pretrainer = OLMoPretrainer(
    pretrain_model,
    output_dir="./olmo_molecular_pretrain",
    learning_rate=1e-4,
    batch_size=2,
    grad_accum_steps=4,
    max_epochs=3,
    save_every_n_steps=50,
    log_every_n_steps=10,
    bf16=(DTYPE == "bfloat16"),
)

### 6.3 Run continued pre-training

In [ ]:
losses = pretrainer.train_on_smiles(
    smiles=sample_corpus,
    max_length=128,
    mol_prefix="SMILES: ",
)

print(f"\nPre-training complete. Steps: {len(losses)}")
if losses:
    print(f"Final loss: {losses[-1]:.4f}")

### 6.4 Generate molecules with the domain-adapted model

In [ ]:
pretrained_molecules = pretrain_model.generate_molecules(
    prompt="",
    n_molecules=8,
    max_new_tokens=80,
    temperature=0.9,
)

print("Molecules from domain-adapted model:")
for i, m in enumerate(pretrained_molecules, 1):
    print(f"  {i}. {m}")

---
## 7. Saving and Loading

### 7.1 Save LoRA adapter (classification model)

In [ ]:
cls_model.save_lora_adapter("./olmo_bbbp_lora")
print("LoRA adapter saved to ./olmo_bbbp_lora")

### 7.2 Load LoRA adapter into a fresh model

In [ ]:
fresh_cls_model = OLMoModel(
    task="classification",
    model_name=MODEL_NAME,
    n_tasks=1,
    n_classes=2,
    use_lora=True,
    lora_r=8,
    lora_alpha=16,
    torch_dtype=DTYPE,
)
fresh_cls_model.load_lora_adapter("./olmo_bbbp_lora")
print("LoRA adapter loaded successfully.")

---
## 8. Summary

In this tutorial we demonstrated:

| Task | Class | Key feature |
|---|---|---|
| Generation | `OLMoModel(task="generation")` | `generate_molecules()` |
| Pre-training | `OLMoPretrainer` | Causal-LM on SMILES corpus |
| Classification | `OLMoModel(task="classification")` + `OLMoFineTuner` | BCE loss, AUROC metric |
| Regression | `OLMoModel(task="regression")` + `OLMoFineTuner` | Huber loss, RMSE metric |
| LoRA | `use_lora=True` | Reduces trainable params by ~100× |

### Next steps

- Scale to `allenai/OLMo-7B` with `torch_dtype="bfloat16"` and `device_map="auto"`  
- Pre-train on ZINC-20 (250 M SMILES) for 1–2 epochs  
- Benchmark on MoleculeNet tasks against ChemBERTa, MolBERT, Uni-Mol  
- Explore SELFIES / IUPAC representations as alternatives to SMILES  
- Try instruction-tuning with `allenai/OLMo-7B-Instruct`